# Bitcoin Forecast Freeze and Validation

## Role
Formal boundary between expensive generation and inexpensive analysis.

## Inputs
All frozen Bitcoin point vectors and the canonical target.

## Outputs
Validation PASS table; no forecast writes.

## Depends On
01–06.

## Authoritative Status
AUTHORITATIVE VALIDATION GATE

## What This Notebook Does Not Do
It does not train, regenerate, promote, or overwrite forecasts.


In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.bitcoin_pipeline import *
RUN_GENERATION = False
PROMOTE_TO_AUTHORITATIVE = False


In [2]:
_,target=load_bitcoin_target(ROOT); train,test=canonical_split(target); v=load_validated_forecasts(ROOT); checks={'1,061 rows and 10 data columns after Timestamp index':v.shape==(1061,10),'Exact timestamps':v.index.equals(test.index),'Actual identity':np.allclose(v.Actual,test),'Complete':not v.isna().any().any(),'Finite':np.isfinite(v).all().all(),'Unique timestamps':v.index.is_unique,'No duplicate vectors':not v.drop(columns='Actual').T.duplicated().any(),'PE-Transformer exists':'Persistence_Enhanced_Transformer' in v}; pd.DataFrame({'Check':checks.keys(),'PASS':checks.values()})

,Check,PASS
0,"1,061 rows and 10 data columns after Timestamp...",True
1,Exact timestamps,True
2,Actual identity,True
3,Complete,True
4,Finite,True
5,Unique timestamps,True
6,No duplicate vectors,True
7,PE-Transformer exists,True


In [3]:
import subprocess,sys; run=subprocess.run([sys.executable,str(ROOT/'src'/'verify_research_artifacts.py')],capture_output=True,text=True); print(run.stdout.splitlines()[-1]); assert run.returncode==0

SUMMARY: 281 PASS, 0 FAIL


## Promotion controls
Generation must target `results/staging/bitcoin/<run-id>/`. `promote_staged_forecast` requires schema, timestamp, row-count, finite-value, optional hash checks, and `explicit_opt_in=True`.